In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 04
# Learning to Say "I Don't Know": Uncertainty as a First-Class Citizen
# =============================================================
#
# Series:  Humble Model / Adversarial Awareness
# Dataset: MNIST (in-distribution), Fashion-MNIST (OOD)
# Model:   Small CNN (same as Notes 01–03)
#
# Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (MNIST + Fashion-MNIST)
#   Part C: Model Architecture (same as previous)
#   Part D: Baseline Model (load from checkpoint or train)
#   Part E: Monte Carlo Dropout
#   Part F: Deep Ensemble
#   Part G: Out-of-Distribution (OOD) Detection
#   Part H: Decision Gate
#   Part I: Putting It All Together — The Humble Model
#   Part J: Evaluation and Summary
# =============================================================

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part A: Imports and Setup
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import copy
import random

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:

# ─────────────────────────────────────────────────────────────
# Part B: Dataset Loading
# ─────────────────────────────────────────────────────────────

# MNIST (in-distribution)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

# Fashion-MNIST (out-of-distribution)
fashion_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))  # Fashion-MNIST stats
])

fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=fashion_transform)

# DataLoaders
train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False, num_workers=0)
ood_loader = DataLoader(fashion_test, batch_size=64, shuffle=False, num_workers=0)

print(f"MNIST train: {len(mnist_train)}, MNIST test: {len(mnist_test)}")
print(f"Fashion-MNIST test (OOD): {len(fashion_test)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part C: Model Architecture (Same as Previous)
# ─────────────────────────────────────────────────────────────

class CNN(nn.Module):
    """Small CNN for MNIST — same architecture as Notes 01–03"""
    
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.fc1: nn.Linear | None = None
        self.fc2 = nn.Linear(128, num_classes)
        self.flattened_size: int | None = None

    def reset_for_input_size(self, input_size: int) -> None:
        """Compute fc1 input dimension from a dry-run on a dummy tensor."""
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_size, input_size)
            x = F.max_pool2d(F.relu(self.conv1(dummy)), 2)
            x = F.max_pool2d(F.relu(self.conv2(x)), 2)
            self.flattened_size = x.view(1, -1).size(1)
            self.fc1 = nn.Linear(self.flattened_size, 128)
        print(f"🔄 Model initialised for {input_size}×{input_size} → flattened: {self.flattened_size}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.fc1 is None:
            raise RuntimeError("Call reset_for_input_size() before forward().")
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

    def forward_with_features(self, x: torch.Tensor):
        """Return both logits and feature representation (before fc2)."""
        if self.fc1 is None:
            raise RuntimeError("Call reset_for_input_size() before forward().")
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        features = x
        logits = self.fc2(x)
        return logits, features

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part D: Baseline Model (Load or Train)
# ─────────────────────────────────────────────────────────────

def train_baseline(model, train_loader, epochs=10):
    """Standard training loop."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"📊 Epoch {epoch} — avg loss: {avg_loss:.4f}")
    
    return model

# Load or train baseline
baseline_model = CNN().to(DEVICE)
baseline_model.reset_for_input_size(28)

if os.path.exists('checkpoint_baseline_essay4.pth'):
    checkpoint = torch.load('checkpoint_baseline_essay4.pth', map_location=DEVICE)
    baseline_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded baseline model from checkpoint")
else:
    print("🆕 Training baseline model from scratch...")
    baseline_model = train_baseline(baseline_model, train_loader, epochs=10)
    torch.save({'model_state_dict': baseline_model.state_dict()}, 'checkpoint_baseline_essay4.pth')
    print("💾 Saved baseline model checkpoint")

# Evaluate clean accuracy
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

clean_acc = evaluate_accuracy(baseline_model, test_loader)
print(f"\n📈 Baseline clean test accuracy: {clean_acc:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part E: Monte Carlo Dropout
# ─────────────────────────────────────────────────────────────

class CNNWithDropout(nn.Module):
    """Same as CNN but with dropout in fc1 and fc2."""
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.fc1 = None
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        self.flattened_size = None

    def reset_for_input_size(self, input_size):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_size, input_size)
            x = F.max_pool2d(F.relu(self.conv1(dummy)), 2)
            x = F.max_pool2d(F.relu(self.conv2(x)), 2)
            self.flattened_size = x.view(1, -1).size(1)
            self.fc1 = nn.Linear(self.flattened_size, 128)
        print(f"🔄 Model with dropout initialised for {input_size}×{input_size}")

    def forward(self, x):
        if self.fc1 is None:
            raise RuntimeError("Call reset_for_input_size() first.")
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# Train a model with dropout
dropout_model = CNNWithDropout(dropout_rate=0.3).to(DEVICE)
dropout_model.reset_for_input_size(28)

if os.path.exists('checkpoint_dropout_model.pth'):
    checkpoint = torch.load('checkpoint_dropout_model.pth', map_location=DEVICE)
    dropout_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded dropout model from checkpoint")
else:
    print("🆕 Training dropout model from scratch...")
    dropout_model = train_baseline(dropout_model, train_loader, epochs=10)
    torch.save({'model_state_dict': dropout_model.state_dict()}, 'checkpoint_dropout_model.pth')
    print("💾 Saved dropout model checkpoint")

def mc_dropout_predict(model, image, num_passes=50):
    """Run MC Dropout inference."""
    model.train()  # Enable dropout
    predictions = []
    for _ in range(num_passes):
        with torch.no_grad():
            pred = torch.softmax(model(image), dim=1)
            predictions.append(pred)
    model.eval()
    predictions = torch.stack(predictions)  # [N, 1, 10]
    mean_pred = predictions.mean(dim=0)
    variance = predictions.var(dim=0)
    return mean_pred, variance

# Test MC Dropout on a few images
test_image, test_label = mnist_test[0]
test_image = test_image.unsqueeze(0).to(DEVICE)

mean_pred, variance = mc_dropout_predict(dropout_model, test_image, num_passes=50)
print(f"\n📊 MC Dropout results on clean MNIST image:")
print(f"   True label: {test_label}")
print(f"   Predicted: {mean_pred.argmax().item()}")
print(f"   Confidence: {mean_pred.max().item():.4f}")
print(f"   Variance (max): {variance.max().item():.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part F: Deep Ensemble
# ─────────────────────────────────────────────────────────────

def train_ensemble_member(seed, train_loader, epochs=10):
    """Train a single ensemble member with a given seed."""
    set_seed(seed)
    model = CNN().to(DEVICE)
    model.reset_for_input_size(28)
    model = train_baseline(model, train_loader, epochs=epochs)
    return model

NUM_ENSEMBLE = 5
ensemble_models = []

for i in range(NUM_ENSEMBLE):
    ckpt_path = f'checkpoint_ensemble_{i}.pth'
    if os.path.exists(ckpt_path):
        model = CNN().to(DEVICE)
        model.reset_for_input_size(28)
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        ensemble_models.append(model)
        print(f"✅ Loaded ensemble member {i}")
    else:
        print(f"🆕 Training ensemble member {i}...")
        model = train_ensemble_member(i, train_loader, epochs=10)
        torch.save({'model_state_dict': model.state_dict()}, ckpt_path)
        ensemble_models.append(model)
        print(f"💾 Saved ensemble member {i}")

def ensemble_predict(models, image):
    """Run ensemble inference."""
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            pred = torch.softmax(model(image), dim=1)
            predictions.append(pred)
    predictions = torch.stack(predictions)  # [M, 1, 10]
    mean_pred = predictions.mean(dim=0)
    variance = predictions.var(dim=0)
    return mean_pred, variance

# Test ensemble
mean_pred_ens, variance_ens = ensemble_predict(ensemble_models, test_image)
print(f"\n📊 Deep Ensemble results on clean MNIST image:")
print(f"   True label: {test_label}")
print(f"   Predicted: {mean_pred_ens.argmax().item()}")
print(f"   Confidence: {mean_pred_ens.max().item():.4f}")
print(f"   Variance (max): {variance_ens.max().item():.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part G: Out-of-Distribution (OOD) Detection
# ─────────────────────────────────────────────────────────────

def extract_features(model, loader, max_samples=1000):
    """Extract features (before fc2) from a data loader."""
    model.eval()
    features = []
    labels = []
    with torch.no_grad():
        for images, lbls in loader:
            images = images.to(DEVICE)
            _, feat = model.forward_with_features(images)
            features.append(feat.cpu())
            labels.extend(lbls.numpy())
            if len(features) * images.size(0) >= max_samples:
                break
    features = torch.cat(features, dim=0)[:max_samples]
    return features, np.array(labels[:max_samples])

# Extract training features for OOD baseline
train_features, train_labels = extract_features(baseline_model, train_loader, max_samples=1000)
print(f"Extracted {len(train_features)} training features.")

def compute_ood_score(feature, training_features):
    """Minimum distance to any training feature."""
    dist = torch.cdist(feature.unsqueeze(0), training_features).min().item()
    return dist

# Test OOD detection on clean, adversarial, and OOD data
# Clean MNIST
clean_image, _ = mnist_test[0]
clean_image = clean_image.unsqueeze(0).to(DEVICE)
_, clean_feat = baseline_model.forward_with_features(clean_image)
clean_ood_score = compute_ood_score(clean_feat.cpu(), train_features)

# Fashion-MNIST (OOD)
ood_image, _ = fashion_test[0]
ood_image = ood_image.unsqueeze(0).to(DEVICE)
_, ood_feat = baseline_model.forward_with_features(ood_image)
ood_score = compute_ood_score(ood_feat.cpu(), train_features)

print(f"\n📊 OOD Detection:")
print(f"   Clean MNIST distance: {clean_ood_score:.4f}")
print(f"   Fashion-MNIST distance: {ood_score:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part H: Decision Gate
# ─────────────────────────────────────────────────────────────

def decision_gate(models_or_model, image, method='ensemble', 
                  threshold_variance=0.05, threshold_confidence=0.7,
                  num_passes=50):
    """
    Decision gate for humble model.
    
    Args:
        models_or_model: Either a single model (for MC Dropout) or list of models (for ensemble)
        method: 'mc_dropout' or 'ensemble'
        threshold_variance: Maximum allowed variance (epistemic uncertainty)
        threshold_confidence: Minimum allowed confidence
        num_passes: Number of MC Dropout passes (if method='mc_dropout')
    
    Returns:
        decision: 'PREDICT' or 'DEFER'
        pred_class: predicted class (or None if deferred)
        uncertainty: variance or mutual information
        confidence: mean confidence
    """
    if method == 'mc_dropout':
        mean_pred, variance = mc_dropout_predict(models_or_model, image, num_passes)
    elif method == 'ensemble':
        mean_pred, variance = ensemble_predict(models_or_model, image)
    else:
        raise ValueError("method must be 'mc_dropout' or 'ensemble'")
    
    confidence = mean_pred.max().item()
    uncertainty = variance.max().item()  # or mutual information
    
    if uncertainty > threshold_variance or confidence < threshold_confidence:
        return 'DEFER', None, uncertainty, confidence
    else:
        return 'PREDICT', mean_pred.argmax().item(), uncertainty, confidence

# Test decision gate on clean, adversarial, and OOD images
# Clean image
decision, pred, unc, conf = decision_gate(
    ensemble_models, clean_image, method='ensemble',
    threshold_variance=0.05, threshold_confidence=0.7
)
print(f"\n📊 Decision Gate on clean MNIST:")
print(f"   Decision: {decision}")
print(f"   Predicted: {pred}")
print(f"   Confidence: {conf:.4f}")
print(f"   Uncertainty: {unc:.4f}")

# OOD image
decision_ood, pred_ood, unc_ood, conf_ood = decision_gate(
    ensemble_models, ood_image, method='ensemble',
    threshold_variance=0.05, threshold_confidence=0.7
)
print(f"\n📊 Decision Gate on OOD (Fashion-MNIST):")
print(f"   Decision: {decision_ood}")
print(f"   Predicted: {pred_ood}")
print(f"   Confidence: {conf_ood:.4f}")
print(f"   Uncertainty: {unc_ood:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part I: Putting It All Together — The Humble Model
# ─────────────────────────────────────────────────────────────

def evaluate_humble_model(models, loader, method='ensemble', threshold_variance=0.05, threshold_confidence=0.7):
    """Evaluate the humble model on a dataset."""
    decisions = []
    preds = []
    uncertainties = []
    confidences = []
    true_labels = []
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        for i in range(images.size(0)):
            image = images[i].unsqueeze(0)
            decision, pred, unc, conf = decision_gate(
                models, image, method=method,
                threshold_variance=threshold_variance,
                threshold_confidence=threshold_confidence
            )
            decisions.append(decision)
            preds.append(pred)
            uncertainties.append(unc)
            confidences.append(conf)
            true_labels.append(labels[i].item())
    
    # Convert to numpy arrays
    decisions = np.array(decisions)
    preds = np.array(preds)
    true_labels = np.array(true_labels)
    
    # Metrics
    total = len(decisions)
    predicted_mask = (decisions == 'PREDICT')
    deferred_mask = (decisions == 'DEFER')
    
    n_predicted = predicted_mask.sum()
    n_deferred = deferred_mask.sum()
    
    if n_predicted > 0:
        pred_correct = (preds[predicted_mask] == true_labels[predicted_mask]).sum()
        accuracy = 100 * pred_correct / n_predicted
    else:
        accuracy = np.nan
    
    coverage = 100 * n_predicted / total
    deferral_rate = 100 * n_deferred / total
    risk = 100 * (1 - accuracy / 100) if not np.isnan(accuracy) else np.nan
    
    return {
        'accuracy': accuracy,
        'coverage': coverage,
        'deferral_rate': deferral_rate,
        'risk': risk,
        'n_predicted': n_predicted,
        'n_deferred': n_deferred,
        'total': total
    }

# Evaluate humble model on clean MNIST
print("\n" + "="*55)
print("🧭 Humble Model Evaluation — Clean MNIST")
print("="*55)

humble_results_clean = evaluate_humble_model(
    ensemble_models, test_loader, method='ensemble',
    threshold_variance=0.05, threshold_confidence=0.7
)

print(f"Accuracy (on predicted): {humble_results_clean['accuracy']:.2f}%")
print(f"Coverage: {humble_results_clean['coverage']:.2f}%")
print(f"Deferral Rate: {humble_results_clean['deferral_rate']:.2f}%")
print(f"Risk (silent failures): {humble_results_clean['risk']:.2f}%")

# Evaluate humble model on OOD (Fashion-MNIST)
print("\n" + "="*55)
print("🧭 Humble Model Evaluation — OOD (Fashion-MNIST)")
print("="*55)

humble_results_ood = evaluate_humble_model(
    ensemble_models, ood_loader, method='ensemble',
    threshold_variance=0.05, threshold_confidence=0.7
)

print(f"Accuracy (on predicted): {humble_results_ood['accuracy']:.2f}%")
print(f"Coverage: {humble_results_ood['coverage']:.2f}%")
print(f"Deferral Rate: {humble_results_ood['deferral_rate']:.2f}%")
print(f"Risk (silent failures): {humble_results_ood['risk']:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part J: Summary and Outputs
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("📋 Experiment Summary")
print("="*55)

print(f"""
Baseline Model:
  - Clean accuracy: {clean_acc:.2f}%

MC Dropout:
  - Example prediction confidence: {mean_pred.max().item():.4f}
  - Example uncertainty (variance): {variance.max().item():.4f}

Deep Ensemble:
  - Example prediction confidence: {mean_pred_ens.max().item():.4f}
  - Example uncertainty (variance): {variance_ens.max().item():.4f}

OOD Detection:
  - Clean MNIST distance: {clean_ood_score:.4f}
  - Fashion-MNIST distance: {ood_score:.4f}

Humble Model (Clean MNIST):
  - Accuracy: {humble_results_clean['accuracy']:.2f}%
  - Coverage: {humble_results_clean['coverage']:.2f}%
  - Deferral Rate: {humble_results_clean['deferral_rate']:.2f}%
  - Risk: {humble_results_clean['risk']:.2f}%

Humble Model (OOD):
  - Accuracy: {humble_results_ood['accuracy']:.2f}%
  - Coverage: {humble_results_ood['coverage']:.2f}%
  - Deferral Rate: {humble_results_ood['deferral_rate']:.2f}%
  - Risk: {humble_results_ood['risk']:.2f}%

Output files:
  - checkpoint_baseline_essay4.pth
  - checkpoint_dropout_model.pth
  - checkpoint_ensemble_*.pth
""")

print("\n✅ Notebook complete!")